# 🏆 Treino EFETIVO v2 - self-play **paralelo** medido por **bb/100**

> **ARQUIVADO / NÃO EXECUTAR.** Este treino foi superado pelo protocolo populacional revisado e publicava o canônico sem gate independente.

**Celula unica - Run all (GPU).** Versao rapida e isolada:

- **N mesas em paralelo** (`SubprocVecEnv`) -> bem mais rapido.
- **Logs isolados** (`selfplay_log.csv` / `SELFPLAY_STATUS.md`) -> nao colidem com outras runs.
- **bb/100 contra bots fixos** = a metrica real. Sem PokerBench. Oponente = snapshot congelado.
- Salva o **MELHOR** modelo no HF automaticamente.

Registro histórico não executável; nenhuma credencial deve ser configurada.

In [ ]:
# ============================================================
# TREINO EFETIVO v2 - self-play PURO, PARALELO, medido por bb/100
# Logs isolados (nao colidem com outras runs). N mesas em paralelo (SubprocVecEnv).
# Registro historico bloqueado; sem clone ou publicacao.
# ============================================================
import os
import re
import subprocess
import sys
ARCHIVED_NOTEBOOK = True
PUBLISH_ARTIFACTS = os.environ.get('POKER_PUBLISH_ARTIFACTS') == '1'
raise RuntimeError('ARQUIVADO: self-play v2 bloqueado; use o pipeline revisado')
from google.colab import userdata
HF_REPO_ID = userdata.get('HF_REPO_ID')
HF_REPO_ID_PATTERN = r'[A-Za-z0-9](?:[A-Za-z0-9._-]{0,94}[A-Za-z0-9])?/[A-Za-z0-9](?:[A-Za-z0-9._-]{0,94}[A-Za-z0-9])?'
if (not isinstance(HF_REPO_ID, str) or not HF_REPO_ID.isascii()
        or len(HF_REPO_ID) > 96 or re.fullmatch(HF_REPO_ID_PATTERN, HF_REPO_ID) is None
        or '..' in HF_REPO_ID or '--' in HF_REPO_ID):
    raise RuntimeError('HF_REPO_ID deve ser ASCII owner/repo canonico.')
# Clone privado legado removido: use um checkout pinado fora deste notebook arquivado.
safe_env_names = ('PATH', 'HOME', 'LANG', 'LC_ALL', 'TMPDIR', 'SYSTEMROOT', 'WINDIR', 'USERPROFILE', 'TEMP', 'TMP')
pip_env = {name: os.environ[name] for name in safe_env_names if name in os.environ}
pip_env.update({'PIP_INDEX_URL': 'https://pypi.org/simple', 'PIP_CONFIG_FILE': os.devnull})
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'poker-arena/backend',
                'stable-baselines3==2.7.0', 'sb3-contrib==2.7.0',
                'gymnasium==1.2.1', 'onnx==1.18.0', 'onnxruntime==1.27.0',
                'huggingface_hub==0.34.3'], check=True, env=pip_env)  # noqa: S603
del pip_env
print('Notebook arquivado; nenhum clone ou upload foi executado.')

import time
import numpy as np
import torch
import gymnasium as gym
import onnxruntime as ort
from gymnasium import spaces
from sb3_contrib import MaskablePPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import SubprocVecEnv
from huggingface_hub import HfApi
import stable_baselines3 as _sb3
import sb3_contrib as _sbc
from poker_arena.ml import FEATURE_SIZE, N_ACTIONS, SelfPlayEnv, encode, eval_bb100
from poker_arena.ml.encoder import legal_mask, to_action
print(f'versoes -> sb3={_sb3.__version__}  sb3-contrib={_sbc.__version__}  torch={torch.__version__}')


def box(titulo, linhas):
    w = max([len(titulo)] + [len(s) for s in linhas]) if linhas else len(titulo)
    print('\n  +' + '-'*(w+2) + '+'); print('  | ' + titulo.ljust(w) + ' |')
    print('  +' + '-'*(w+2) + '+')
    for s in linhas: print('  | ' + s.ljust(w) + ' |')
    print('  +' + '-'*(w+2) + '+')

def barra(frac, n=22):
    frac = max(0.0, min(1.0, frac)); c = int(frac*n)
    return '[' + '#'*c + '.'*(n-c) + f'] {100*frac:3.0f}%'


# -------------------- CONFIG --------------------
N_ENVS = 8                # mesas em paralelo (acelera ~Nx). Baixe se faltar CPU/RAM.
TOTAL_STEPS = 1_000_000   # passos TOTAIS (somados entre as mesas)
EVAL_EVERY = 100_000      # mede bb/100 + atualiza o oponente congelado
EVAL_HANDS = 500
LOG_CSV = 'selfplay_log.csv'      # nomes PROPRIOS -> nao colidem com outras runs
STATUS_MD = 'SELFPLAY_STATUS.md'

box('TREINO EFETIVO v2 - self-play PURO, PARALELO, bb/100', [
    f'{N_ENVS} mesas em paralelo, contra um SNAPSHOT congelado do agente.',
    'Metrica = bb/100 contra bots fixos (o objetivo REAL). Sem PokerBench.',
    'Logs isolados no HF (selfplay_log.csv / SELFPLAY_STATUS.md).',
])

# -------------------- env (com troca de oponente p/ subprocessos) --------------------
class GymPokerEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.env = SelfPlayEnv(n_players=6)
        self.observation_space = spaces.Box(0.0, 5.0, (FEATURE_SIZE,), dtype=np.float32)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self._m = [True]*N_ACTIONS
        self._sess = None
    def reset(self, *, seed=None, options=None):
        o, self._m = self.env.reset(); return np.asarray(o, np.float32), {}
    def step(self, a):
        o, self._m, r, d = self.env.step(int(a)); return np.asarray(o, np.float32), float(r), bool(d), False, {}
    def action_masks(self):
        return np.asarray(self._m, bool)
    def set_opponent_path(self, path):
        sess = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
        iname = sess.get_inputs()[0].name
        def opp(feats, mask):
            logits = sess.run(None, {iname: np.asarray([feats], np.float32)})[0][0]
            legal = np.array([i for i, ok in enumerate(mask) if ok])
            z = logits[legal]; z = z - z.max(); p = np.exp(z); p = p/p.sum()
            return int(np.random.choice(legal, p=p))
        self.env.set_opponent(opp); self._sess = sess

def make_env():
    return GymPokerEnv()

# VecEnv criado ANTES do modelo (fork antes do CUDA) -> seguro
vec_env = SubprocVecEnv([make_env for _ in range(N_ENVS)], start_method='fork')
model = MaskablePPO('MlpPolicy', vec_env, n_steps=512, batch_size=512, ent_coef=0.01,
                    gamma=0.99, gae_lambda=0.95, policy_kwargs=dict(net_arch=[256, 256]),
                    verbose=0, device='auto')
hf_token = userdata.get('HF_TOKEN')
if not isinstance(hf_token, str) or not hf_token:
    raise RuntimeError('HF_TOKEN e obrigatorio somente no instante de publicar.')
api = HfApi(token=hf_token)
hf_token = ''
api.create_repo(HF_REPO_ID, repo_type='model', exist_ok=True, private=True)


def export_onnx(path):
    dev = model.device
    model.policy.to('cpu').eval()
    class _Onnx(torch.nn.Module):
        def __init__(s):
            super().__init__()
            s.fe, s.mlp, s.an = model.policy.features_extractor, model.policy.mlp_extractor, model.policy.action_net
        def forward(s, obs):
            f = s.fe(obs); lp, _ = s.mlp(f); return s.an(lp)
    torch.onnx.export(_Onnx(), torch.zeros(1, FEATURE_SIZE), path, input_names=['obs'],
                      output_names=['logits'], dynamic_axes={'obs': {0: 'batch'}, 'logits': {0: 'batch'}},
                      opset_version=17, dynamo=False)
    model.policy.to(dev).train()

def onnx_hero(sess):
    iname = sess.get_inputs()[0].name
    def hero(obs):
        logits = sess.run(None, {iname: np.asarray([encode(obs)], np.float32)})[0][0].tolist()
        mask = legal_mask(obs)
        best = max((i for i in range(N_ACTIONS) if mask[i]), key=lambda i: logits[i])
        return to_action(obs, best)
    return hero


class Treino(BaseCallback):
    def __init__(self):
        super().__init__(); self._last = 0; self._t0 = None; self._best = -1e9; self._rows = []
    def _on_training_start(self):
        self._t0 = time.time()
        box('PAINEL AO VIVO  (bb/100 = forca de verdade)', [
            'bb/100 vs random ...... deve ficar BEM positivo (bate os fracos)',
            'bb/100 vs heuristico ... O ALVO: subir rumo a POSITIVO = aprendeu a ganhar',
            '',
            'O melhor modelo (maior bb/100 vs heuristico) e salvo no HF automaticamente.',
        ])
        box('NAO TEM PERGUNTA IDIOTA', [
            'P: Por que medir bb/100 e nao "concordancia com solver"?',
            'R: Um modelo pode concordar com o solver e mesmo assim PERDER no jogo.',
            '   bb/100 mede o que importa: ganhar fichas contra adversarios reais.',
        ])
        print(f'   >> logs ao vivo: https://huggingface.co/{HF_REPO_ID}  ({STATUS_MD} / {LOG_CSV})')
    def _hf(self, n, pct, sps, eta, br, bh):
        self._rows.append((n, round(pct, 1), round(sps), round(eta, 1), round(br, 1), round(bh, 1), round(time.time()-self._t0)))
        lines = ['passos,pct,maos_s,eta_min,bb100_random,bb100_heur,segundos'] + [','.join(map(str, r)) for r in self._rows]
        open(LOG_CSV, 'w').write('\n'.join(lines) + '\n')
        md = ['# Poker Arena - treino EFETIVO v2 (bb/100 ao vivo)', '',
              f'**Passo {n:,} de {TOTAL_STEPS:,}**  ({pct:.0f}%)', '',
              f'`{barra(n/TOTAL_STEPS)}`  -  faltam ~{eta:.0f} min  -  {sps:.0f} maos/s', '',
              '| metrica | valor |', '|---|---|',
              f'| bb/100 vs random | {br:+.1f} |',
              f'| bb/100 vs heuristico | {bh:+.1f} |',
              f'| melhor vs heuristico | {max(self._best, bh):+.1f} |', '',
              '> bb/100 vs heuristico subindo = aprendizado real.']
        open(STATUS_MD, 'w').write('\n'.join(md))
        try:
            api.upload_file(path_or_fileobj=LOG_CSV, path_in_repo=LOG_CSV, repo_id=HF_REPO_ID, repo_type='model')
            api.upload_file(path_or_fileobj=STATUS_MD, path_in_repo=STATUS_MD, repo_id=HF_REPO_ID, repo_type='model')
        except Exception:
            print('   (aviso: upload pro HF falhou desta vez - segue treinando)')
    def _on_step(self):
        n = self.num_timesteps
        if n - self._last >= EVAL_EVERY:
            self._last = n
            el = max(time.time()-self._t0, 1e-9); sps = n/el
            eta = (TOTAL_STEPS-n)/sps/60 if sps > 0 else 0.0
            export_onnx('current.onnx')
            sess = ort.InferenceSession('current.onnx', providers=['CPUExecutionProvider'])
            br = eval_bb100(onnx_hero(sess), 'random', hands=EVAL_HANDS, seed=999)
            bh = eval_bb100(onnx_hero(sess), 'heuristic', hands=EVAL_HANDS, seed=999)
            self.training_env.env_method('set_opponent_path', 'current.onnx')  # atualiza TODAS as mesas
            self.model.policy.set_training_mode(True)
            print()
            print(f'  {barra(n/TOTAL_STEPS)}   faltam ~{eta:.0f} min   ({sps:.0f} maos/s)')
            print(f'     bb/100 vs random: {br:+6.1f}    vs heuristico: {bh:+6.1f}'
                  + ('   <<< NOVO MELHOR!' if bh > self._best else ''))
            self._hf(n, 100*n/TOTAL_STEPS, sps, eta, br, bh)
            if bh > self._best:
                self._best = bh
                export_onnx('poker_expert.onnx')
                try:
                    api.upload_file(path_or_fileobj='poker_expert.onnx', path_in_repo='poker_expert.onnx', repo_id=HF_REPO_ID, repo_type='model')
                except Exception as exc:
                    print(f'aviso: upload do checkpoint falhou ({type(exc).__name__}); artefato local preservado')
        return True


box('TREINANDO (self-play paralelo)...', [
    f'{N_ENVS} mesas. Oponente comeca aleatorio e vira snapshot do agente a cada eval.',
])
cb = Treino()
model.learn(total_timesteps=TOTAL_STEPS, callback=cb)
box('TREINO CONCLUIDO', [
    f'Melhor bb/100 vs heuristico: {cb._best:+.1f}',
    'poker_expert.onnx (o MELHOR) esta no seu HF, pronto pro MLBot.',
])
